# Week 1: One Toolbox, Three Framings
## Supervised, unsupervised and reinforcement learning use the same models

You have just seen **linear regression** in supervised learning. The point of this short notebook:

> The *model* (a linear regression, a neural network) is the same tool in all three settings. What differs is the **framing**: what data you have, what signal you learn from, and who decides which data you get to see.

We use one hidden function, $g(x) = \sin(x)$, and the same two model families everywhere:
- **Linear regression** on polynomial features (linear in its parameters, like the regression you already know)
- A small **neural network** (MLP)

Each section changes only the framing. A summary table at the end brings it together.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)  # keep the output clean
rng = np.random.default_rng(0)

X_MAX, NOISE = 2 * np.pi, 0.2
x_grid = np.linspace(0, X_MAX, 200)

def g(x):
    """The hidden truth shared by all three sections."""
    return np.sin(x)

KINDS = ["linear", "mlp"]
NAMES = {"linear": "Linear regression (degree-5 features)", "mlp": "Neural network (MLP)"}

def make_model(kind):
    """A regression model x -> y. Used for supervised learning and for RL."""
    if kind == "linear":
        return make_pipeline(PolynomialFeatures(5), LinearRegression())
    return make_pipeline(StandardScaler(),
                         MLPRegressor(hidden_layer_sizes=(16, 16), activation="tanh", alpha=0.1,
                                      solver="lbfgs", max_iter=2000, random_state=0))

def make_autoencoder(kind):
    """A model that reconstructs its input through a 1-unit bottleneck. Used for unsupervised learning."""
    activation = "identity" if kind == "linear" else "tanh"
    return MLPRegressor(hidden_layer_sizes=(16, 1, 16), activation=activation,
                        solver="lbfgs", max_iter=3000, random_state=0)

## 1. Supervised learning: learn from the right answers

- **Given:** inputs $x$ together with the correct output $y$ (labels). Here: noisy samples of $g$.
- **Learn:** a function $f(x) \approx y$ by minimising the squared error to the labels.
- **Data comes from:** a fixed dataset that somebody collected for us.

In [ ]:
x = rng.uniform(0, X_MAX, 100)
y = g(x) + NOISE * rng.standard_normal(100)          # labeled data: (x, y) pairs

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, kind in zip(axes, KINDS):
    model = make_model(kind).fit(x[:, None], y)       # <- the whole "learning" step: fit(inputs, labels)
    pred = model.predict(x_grid[:, None])
    ax.scatter(x, y, s=12, color="gray", label="labeled data")
    ax.plot(x_grid, g(x_grid), "k--", lw=1, label="truth $g(x)$")
    ax.plot(x_grid, pred, "C3", lw=2, label="fitted $f(x)$")
    ax.set_title(f"{NAMES[kind]}\nerror vs. truth: {np.mean((pred - g(x_grid)) ** 2):.3f}")
    ax.set_xlabel("x"); ax.grid(alpha=0.3)
axes[0].set_ylabel("y"); axes[0].legend(loc="lower left")
plt.tight_layout(); plt.show()

## 2. Unsupervised learning: find structure without labels

Take the *same points* as before, but now nobody tells us which coordinate is "the answer". We only have a cloud of 2D points $(x, y)$.

- **Given:** points, no labels.
- **Learn:** a compact description of the data. Here an **autoencoder**: a model trained to reconstruct its own input after squeezing it through a bottleneck of a *single number*. The training target is the input itself: `fit(P, P)`.
- **Data comes from:** a fixed dataset.

A *linear* autoencoder is exactly **PCA** (regression of the data onto itself through a 1-dimensional bottleneck). A *neural* autoencoder can follow curved structure. Gray lines show the reconstruction error of each point, the analogue of the residuals in regression.

In [ ]:
P = np.c_[x, y]                                       # the same points, now without a "label" column
scaler = StandardScaler().fit(P)
Z = scaler.transform(P)

errors = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, kind in zip(axes, KINDS):
    ae = make_autoencoder(kind).fit(Z, Z)             # <- same fit call, but the target is the input itself
    recon = scaler.inverse_transform(ae.predict(Z))
    errors[kind] = np.mean((recon - P) ** 2)
    ax.scatter(P[:, 0], P[:, 1], s=12, color="gray", label="data (no labels)")
    ax.plot([P[:, 0], recon[:, 0]], [P[:, 1], recon[:, 1]], color="gray", alpha=0.3, lw=0.6)
    ax.scatter(recon[:, 0], recon[:, 1], s=14, color="C3", label="reconstruction (1-D)")
    ax.set_title(f"{'Linear' if kind == 'linear' else 'Neural'} autoencoder\nreconstruction error: {errors[kind]:.3f}")
    ax.set_xlabel("x"); ax.grid(alpha=0.3)
axes[0].set_ylabel("y"); axes[0].legend(loc="lower left")
plt.tight_layout(); plt.show()

pca = PCA(n_components=1).fit(Z)
pca_error = np.mean((scaler.inverse_transform(pca.inverse_transform(pca.transform(Z))) - P) ** 2)
print(f"Reconstruction error  linear autoencoder: {errors['linear']:.3f}   |   PCA with 1 component: {pca_error:.3f}")

## 3. Reinforcement learning: learn from rewards for your own actions

Now there is **no dataset at all**. An agent picks an action $a$ and only sees a noisy reward $r = g(a) + \text{noise}$ for the action it chose. It never sees the correct action, nor the curve $g$. Its goal is to collect as much reward as possible, i.e. to find the best action.

The idea (a bandit with a continuous action, like the lab): keep a regression model $\hat Q(a) \approx$ expected reward, act greedily on it, and explore with probability $\varepsilon$.

- **Given:** nothing up front. The agent interacts with the environment.
- **Learn from:** rewards, a score for what it did rather than the right answer. Again the same `fit` call, now on pairs $(a, r)$.
- **Data comes from:** the agent's own actions, so what it does decides what it gets to learn.

In [ ]:
def run_agent(kind, n_steps=200, epsilon=0.1, refit_every=10):
    actions, rewards, q_model = [], [], None
    for t in range(n_steps):
        if q_model is None or rng.random() < epsilon:        # explore: try a random action
            a = rng.uniform(0, X_MAX)
        else:                                                # exploit: best action according to the model
            a = x_grid[np.argmax(q_model.predict(x_grid[:, None]))]
        r = g(a) + NOISE * rng.standard_normal()             # the environment answers with a reward
        actions.append(a); rewards.append(r)
        if (t + 1) % refit_every == 0:                       # <- same fit call, on the agent's own experience
            q_model = make_model(kind).fit(np.array(actions)[:, None], np.array(rewards))
    return np.array(actions), np.array(rewards), q_model


fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, kind in zip(axes[:2], KINDS):
    actions, rewards, q_model = run_agent(kind)
    ax.scatter(actions, rewards, s=10, color="gray", alpha=0.5, label="experienced $(a, r)$")
    ax.plot(actions, np.full_like(actions, -1.45), "|", color="C0", alpha=0.4, label="actions taken")
    ax.plot(x_grid, g(x_grid), "k--", lw=1, label="truth $g(a)$ (hidden)")
    ax.plot(x_grid, q_model.predict(x_grid[:, None]), "C3", lw=2, label="learned $\\hat Q(a)$")
    ax.set_title(f"{NAMES[kind]}\nlast 100 actions: mean reward {rewards[-100:].mean():.2f}")
    ax.set_xlabel("action a"); ax.set_ylim(-1.6, 1.6); ax.grid(alpha=0.3)
    axes[2].plot(np.cumsum(rewards) / np.arange(1, len(rewards) + 1), label=NAMES[kind])
axes[0].set_ylabel("reward"); axes[0].legend(loc="upper right", fontsize=8)
axes[2].axhline(0, color="k", ls=":", label="random actions (average 0)")
axes[2].axhline(1, color="k", ls="--", lw=1, label="best possible (1)")
axes[2].set_title("Average reward so far"); axes[2].set_xlabel("step"); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

Notice that the agent's samples (blue ticks) pile up near the best action: it collects data where it *thinks* the reward is high. In supervised learning the dataset is fixed in advance, in RL the agent's own choices shape the data it learns from. That is why exploration matters.

### Connection to the Week 1 lab

In the bandit lab you updated `Q[a] += (reward - Q[a]) / N[a]`. That is a regression as well: one gradient step on the squared error $\tfrac12 (r - Q_a)^2$ with step size $1/N_a$, where the model is a **lookup table** with one number per arm. Here we replaced the table by a linear model or a neural network, so that $\hat Q$ can generalise between similar actions. In deep RL, this is the step from tables to networks.

## Summary

| | Supervised | Unsupervised | Reinforcement |
|---|---|---|---|
| **Given** | inputs and correct outputs $(x, y)$ | inputs only | nothing, the agent interacts |
| **Learning signal** | the label $y$ | the input itself (reconstruct it) | a reward for the chosen action |
| **Regression target** | $y$ | $x$ | $r$ (later in the module: $r + \gamma \max_{a'} Q(s', a')$) |
| **Who generates the data** | fixed dataset | fixed dataset | the agent's own actions |
| **Goal** | predict $y$ for new $x$ | find structure, compress | act to maximise reward |
| **Models used here** | linear regression, MLP | linear autoencoder (= PCA), MLP autoencoder | linear regression, MLP |

**Takeaway:** the model is the tool, the paradigm is the framing: what data you have, what signal you learn from, and who controls the data.

### Things to try
- Change `PolynomialFeatures(5)` to degree 1 or 2. What happens in each of the three sections?
- Set `epsilon=0` in `run_agent` and re-run with different seeds. What changes?
- Give the autoencoder a wider bottleneck, `hidden_layer_sizes=(16, 2, 16)`. What should the reconstruction error do?